In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

STORAGE = "stecomlakedev"
SILVER = f"abfss://silver@{STORAGE}.dfs.core.windows.net"
GOLD = f"abfss://gold@{STORAGE}.dfs.core.windows.net"

conn = dbutils.secrets.get("ecomlake", "eventhub-listen-connstr")

GOLD = f"abfss://gold@{STORAGE}.dfs.core.windows.net"

src = (spark.read.format("delta").load(f"abfss://silver@{STORAGE}.dfs.core.windows.net/customers")
    .select(
        F.col("customer_unique_id").alias("customer_id"),
        F.col("customer_city").alias("city"),
        F.col("customer_state").alias("state"))
    .dropDuplicates(["customer_id"]))

dim_path = f"{GOLD}/dim_customer"

if not DeltaTable.isDeltaTable(spark, dim_path):
    (src
      .withColumn("customer_sk", F.monotonically_increasing_id())
      .withColumn("effective_from", F.current_timestamp())
      .withColumn("effective_to", F.lit(None).cast("timestamp"))
      .withColumn("is_current", F.lit(True))
      .write.format("delta").save(dim_path))
else:
    dim = DeltaTable.forPath(spark, dim_path)

    # 1) Find rows whose tracked attributes changed
    changes = (src.alias("s")
        .join(dim.toDF().filter("is_current").alias("d"), "customer_id")
        .filter("s.city <> d.city OR s.state <> d.state")
        .select("s.*"))

    # 2) Expire current versions of changed rows
    (dim.alias("d").merge(
            changes.alias("c"), "d.customer_id = c.customer_id AND d.is_current = true")
        .whenMatchedUpdate(set={
            "is_current": "false",
            "effective_to": "current_timestamp()"})
        .execute())

    # 3) Insert new current versions + brand-new customers
    existing_ids = dim.toDF().select("customer_id").distinct()
    new_customers = src.join(existing_ids, "customer_id", "left_anti")

    (changes.unionByName(new_customers)
        .withColumn("customer_sk", F.monotonically_increasing_id())
        .withColumn("effective_from", F.current_timestamp())
        .withColumn("effective_to", F.lit(None).cast("timestamp"))
        .withColumn("is_current", F.lit(True))
        .write.format("delta").mode("append").save(dim_path))

In [0]:
products = spark.read.format("delta").load(f"{SILVER}/products")

dim_product = (products
    .select(
        F.col("product_id"),
        F.col("category_name"),
        F.col("product_weight_g").alias("weight_g"),
        F.col("product_volume_cm3").alias("volume_cm3"),
        F.col("product_photos_qty").cast("int").alias("photos_qty"))
    .withColumn("size_band",
        F.when(F.col("volume_cm3") < 5000, "small")
         .when(F.col("volume_cm3") < 30000, "medium")
         .otherwise("large")))

(dim_product.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .save(f"{GOLD}/dim_product"))